# **Text Lemmatisation Pipeline**
A preprocessing pipeline for Russian text with support for emojis, smiles and special characters.

## Overview
A complete solution for lemmatising Russian text while preserving meaningful context. Converts emojis, smileys, CAPS words and punctuation into semantic markers for further NLP analysis.

## Key Features
- **Multi-stage processing**: emoji categorisation (200+ emojis grouped into semantic categories like happy, love, anger, etc.), CAPS recognition with lemmatisation, punctuation handling, and morphological analysis
- **Morphological analysis** via pymorphy3 with POS filtering and stopword removal
- **LRU caching** for optimal performance on large datasets
- **Configurable** stopword removal, token length, and POS filters

## Core Components
- **TextLemmatiser Class**: Main processing engine with configurable parameters and built-in caching
- **Configuration System**: Customisable stopwords, POS tags, token length, and language settings
- **Special Token Preservation**: Retains semantic markers for emojis (`EMOJI_HAPPY`), smiles (`SMILE_LAUGH`), CAPS words (`CAPS_lemma`), and punctuation (`PUNC_EXCLAM`)

## Use Cases
- Sentiment analysis of social media posts and reviews
- Topic modelling and text classification
- Preprocessing for machine learning pipelines
- Text normalisation for linguistic analysis

## Technical Details
Built with pymorphy3, NLTK, and pandas. Optimised for large datasets via LRU caching and progress bars (tqdm). Provides a simple API with both class-based and functional interfaces.

## Output
Returns lemmatised text with preserved semantic markers for:
- Emojis: `EMOJI_HAPPY`, `EMOJI_LOVE`, `EMOJI_ANGRY`, etc.
- Smiles: `SMILE_HAPPY`, `SMILE_SAD`, `SMILE_LAUGH`, etc.
- CAPS words: `CAPS_важно` (lemmatised)
- Punctuation: `PUNC_EXCLAM`, `PUNC_QUEST`, `PUNC_ELLIPSIS`, etc.
- Numbers: `123_NUM`

## Quick Start
```python
from text_lemmatiser import TextLemmatiser, lemmatise

# Using the functional interface
result = lemmatise("Привет! Как дела? 😊")

# Using the class interface
lemmatiser = TextLemmatiser()
result = lemmatiser.process("Это ВАЖНО!!!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!wget https://raw.githubusercontent.com/ValeriaMagnatka/surprise_dataset/main/categories.csv

--2026-06-28 07:46:36--  https://raw.githubusercontent.com/ValeriaMagnatka/surprise_dataset/main/categories.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24788 (24K) [text/plain]
Saving to: ‘categories.csv’

categories.csv      100%[===================>]  24.21K  --.-KB/s    in 0s      

2026-06-28 07:46:36 (108 MB/s) - ‘categories.csv’ saved [24788/24788]



In [ ]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 105.8 MB/s eta 0:00:00


In [ ]:
from functools import lru_cache
from typing import Optional, Set, Dict, Tuple, List
from dataclasses import dataclass, field
import logging

from collections import Counter
import pandas as pd
from tqdm import tqdm


import re
import pymorphy3
from pymorphy3 import MorphAnalyzer
morph = MorphAnalyzer()

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk_stop_words = set(stopwords.words('russian'))

# Logging configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Lemmatisation with special tokens

In [ ]:
# Set of marker prefixes used to identify special tokens in output
SPECIAL_MARKERS = {'EMOJI_', 'SMILE_', 'CAPS_', 'PUNC_'}

In [ ]:
# DICTIONARIES
# Map of emojis to semantic categories
EMOJI_MAP = {
    # Positive emotions
    '😊': 'EMOJI_HAPPY', '☺': 'EMOJI_HAPPY', '😁': 'EMOJI_HAPPY',
    '😄': 'EMOJI_HAPPY', '😀': 'EMOJI_HAPPY', '😃': 'EMOJI_HAPPY',
    '☺️': 'EMOJI_HAPPY', '🤗': 'EMOJI_HAPPY',

    # Love and affection
    '❤': 'EMOJI_LOVE', '♥': 'EMOJI_LOVE', '💕': 'EMOJI_LOVE',
    '💗': 'EMOJI_LOVE', '💖': 'EMOJI_LOVE', '💓': 'EMOJI_LOVE',
    '💞': 'EMOJI_LOVE', '💝': 'EMOJI_LOVE', '💘': 'EMOJI_LOVE',
    '🩷': 'EMOJI_LOVE', '🧡': 'EMOJI_LOVE', '💛': 'EMOJI_LOVE',
    '💚': 'EMOJI_LOVE', '💙': 'EMOJI_LOVE', '💜': 'EMOJI_LOVE',
    '🖤': 'EMOJI_LOVE', '🤍': 'EMOJI_LOVE', '🩵': 'EMOJI_LOVE',
    '🫶': 'EMOJI_EMBARRASS',

    # Admiration and excitement
    '😍': 'EMOJI_ADORE', '🥰': 'EMOJI_ADORE', '🤩': 'EMOJI_ADORE',
    '😻': 'EMOJI_ADORE', '🥳': 'EMOJI_ADORE', '⭐': 'EMOJI_ADORE',
    '🌟': 'EMOJI_ADORE', '✨': 'EMOJI_ADORE', '💫': 'EMOJI_ADORE',
    '🎉': 'EMOJI_ADORE', '🎊': 'EMOJI_ADORE',

    # Laughter
    '😂': 'EMOJI_LAUGH', '🤣': 'EMOJI_LAUGH', '😹': 'EMOJI_LAUGH',
    '😆': 'EMOJI_LAUGH', '😅': 'EMOJI_LAUGH',

    # Approval and agreement
    '👍': 'EMOJI_APPROVE', '👌': 'EMOJI_APPROVE', '🙏': 'EMOJI_APPROVE',
    '🤝': 'EMOJI_APPROVE', '✅': 'EMOJI_APPROVE', '✔': 'EMOJI_APPROVE',
    '✔️': 'EMOJI_APPROVE', '☝': 'EMOJI_APPROVE', '☝️': 'EMOJI_APPROVE',
    '✌': 'EMOJI_APPROVE', '✌️': 'EMOJI_APPROVE', '🤘': 'EMOJI_APPROVE',
    '🤙': 'EMOJI_APPROVE', '✊': 'EMOJI_APPROVE', '👏': 'EMOJI_APPROVE',
    '💯': 'EMOJI_APPROVE',

    # Beauty and nature
    '🌹': 'EMOJI_BEAUTY', '🌸': 'EMOJI_BEAUTY', '🌺': 'EMOJI_BEAUTY',
    '🌷': 'EMOJI_BEAUTY', '🌿': 'EMOJI_BEAUTY', '🌼': 'EMOJI_BEAUTY',
    '🍀': 'EMOJI_BEAUTY', '🌲': 'EMOJI_BEAUTY', '🌈': 'EMOJI_BEAUTY',

    # Motivation and achievement
    '💪': 'EMOJI_STRENGTH', '🏆': 'EMOJI_SUCCESS', '🥇': 'EMOJI_SUCCESS',
    '🎁': 'EMOJI_SUCCESS', '💐': 'EMOJI_SUCCESS', '🎈': 'EMOJI_SUCCESS',
    '🎂': 'EMOJI_SUCCESS', '🔥': 'EMOJI_FIRE', '⚡': 'EMOJI_ENERGY',
    '🔝': 'EMOJI_TOP',

    # Negative emotions
    '😢': 'EMOJI_SAD', '😭': 'EMOJI_SAD', '😔': 'EMOJI_SAD',
    '😞': 'EMOJI_SAD', '☹': 'EMOJI_SAD', '☹️': 'EMOJI_SAD',
    '🥺': 'EMOJI_SAD', '🥲': 'EMOJI_SAD', '🫠': 'EMOJI_SAD',
    '💔': 'EMOJI_BROKEN_HEART',

    # Anger
    '😡': 'EMOJI_ANGRY', '🤬': 'EMOJI_ANGRY', '😤': 'EMOJI_ANGRY',
    '💢': 'EMOJI_ANGRY',

    # Disgust
    '🤮': 'EMOJI_DISGUST', '🤢': 'EMOJI_DISGUST', '💩': 'EMOJI_DISGUST',

    # Fear
    '😱': 'EMOJI_FEAR', '😨': 'EMOJI_FEAR', '😰': 'EMOJI_FEAR',

    # Other expressions
    '🤔': 'EMOJI_THINK', '🙄': 'EMOJI_THINK', '🤨': 'EMOJI_THINK',
    '🧐': 'EMOJI_THINK', '😕': 'EMOJI_THINK',
    '😳': 'EMOJI_EMBARRASS', '🫣': 'EMOJI_EMBARRASS',
    '🙈': 'EMOJI_EMBARRASS', '💥': 'EMOJI_EXPLOSIVE', '🚀': 'EMOJI_ROCKET',
    '😜': 'EMOJI_SILLY', '😝': 'EMOJI_SILLY', '🤪': 'EMOJI_SILLY',
    '😋': 'EMOJI_SILLY', '😏': 'EMOJI_SMIRK', '😉': 'EMOJI_WINK',
    '🍕': 'EMOJI_FOOD', '🍺': 'EMOJI_FOOD', '🍻': 'EMOJI_FOOD',
    '🍷': 'EMOJI_FOOD', '🍓': 'EMOJI_FOOD', '🍰': 'EMOJI_FOOD',
    '🍹': 'EMOJI_FOOD', '☕': 'EMOJI_FOOD', '☕️': 'EMOJI_FOOD',
    '♂': 'EMOJI_MAN', '♀': 'EMOJI_WOMAN',
    '🕊': 'EMOJI_PIGEON', '🕊️': 'EMOJI_PIGEON',
    '♂️': 'EMOJI_GESTURE', '♀️': 'EMOJI_GESTURE',
    '👩': 'EMOJI_GESTURE', '👨': 'EMOJI_GESTURE',
    '👆': 'EMOJI_GESTURE', '👉': 'EMOJI_GESTURE',
    '👋': 'EMOJI_GESTURE', '🤌': 'EMOJI_GESTURE',
    '🤟': 'EMOJI_GESTURE', '🫰': 'EMOJI_GESTURE',
    '☀': 'EMOJI_SUN', '☀️': 'EMOJI_SUN', '🌞': 'EMOJI_SUN',
    '🌊': 'EMOJI_WATER', '💣': 'EMOJI_BOMB',
    '💅': 'EMOJI_NAILS', '🧚': 'EMOJI_FAIRY',
    '🤡': 'EMOJI_CLOWN', '🤓': 'EMOJI_NERD',
    '😬': 'EMOJI_GRIMACE', '😵': 'EMOJI_DIZZY',
    '🤯': 'EMOJI_EXPLODE_HEAD', '🤦': 'EMOJI_FACEPALM',
    '🤷': 'EMOJI_SHRUG', '😒': 'EMOJI_UNAMUSED',
    '🤭': 'EMOJI_HUSH', '🙃': 'EMOJI_UPSIDE_DOWN',
    '😌': 'EMOJI_RELIEF', '🙂': 'EMOJI_SLIGHT_SMILE',
    '😇': 'EMOJI_INNOCENT', '💋': 'EMOJI_KISS',
    '💃': 'EMOJI_DANCE', '😎': 'EMOJI_COOL',
    '🥂': 'EMOJI_TOAST', '😐': 'EMOJI_NEUTRAL', '❗': 'EMOJI_ANGRY'
}

# Text Smiles
SMILE_MAP = {
    ':-o': 'SMILE_SURPRISE', ':o': 'SMILE_SURPRISE',
    ':-?': 'SMILE_CONFUSED', ':?': 'SMILE_CONFUSED',
    ':-)': 'SMILE_HAPPY', ':)': 'SMILE_HAPPY',
    ':-(': 'SMILE_SAD', ':(': 'SMILE_SAD',
    ':-d': 'SMILE_LAUGH', ':d': 'SMILE_LAUGH',
    ';-)': 'SMILE_WINK', ';)': 'SMILE_WINK',
    ':-p': 'SMILE_TONGUE', ':p': 'SMILE_TONGUE',
    ':-/': 'SMILE_SKEPTICAL', ':/': 'SMILE_SKEPTICAL',
    ':-|': 'SMILE_NEUTRAL', ':|': 'SMILE_NEUTRAL',
}

In [ ]:
# REGULAR EXPRESSIONS

# Matches only emojis from the dictionary
RE_EMOJI = re.compile(
    '|'.join(re.escape(emoji) for emoji in EMOJI_MAP.keys()),
    flags=re.UNICODE
)

# Improved pattern for detecting text-based smiles
RE_SMILE = re.compile(
    r'(?:[:;]-?[oO?\)\(dDpP/|])(?!\w)',
    flags=re.UNICODE
)

# Detects words in ALL CAPS for special processing
RE_CAPS = re.compile(r'\b([A-ZА-ЯЁ]{2,})\b', flags=re.UNICODE)

# Patterns for handling various punctuation marks
RE_EXCLAM_QUEST = re.compile(r'([!?]){2,}')  # Multiple ! or ?
RE_SINGLE_EXCLAM = re.compile(r'!')          # Single !
RE_SINGLE_QUEST = re.compile(r'\?')          # Single ?
RE_ELLIPSIS = re.compile(r'\.{3,}')          # Ellipsis (...)
RE_OTHER_PUNCT = re.compile(r'[,;:\'"\[\]{}()<>]')  # Other punctuation

# Utility patterns
RE_NUMBER = re.compile(r'\b\d+\b')  # Numbers
RE_SPACE = re.compile(r'\s+')      # Whitespace


In [ ]:
# CONFIGURATION

@dataclass
class Config:
    """Configuration settings for the text lemmatiser."""

    keep_caps: bool = True  # Whether to preserve and mark CAPS words
    remove_stopwords: bool = True  # Whether to remove stop words
    min_token_len: int = 2  # Minimum token length to keep
    cache_size: int = 1000  # Maximum size for LRU caches
    language: str = 'russian'  # Language for stopwords and tokenisation

    # Additional stopwords beyond the default NLTK set
    extra_stopwords: Set[str] = field(default_factory=set)  # Пустое множество

    # Allowed POS tags for lemmatised tokens
    allowed_pos: Set[str] = field(default_factory=lambda: {
        'NOUN', 'VERB', 'INFN', 'ADJF', 'ADJS', 'ADVB', 'PRED'
    })

    log_unknown_emoji: bool = True  # Whether to log unknown emojis

In [ ]:
# LEMMATISER

class TextLemmatiser:
    """
    Main class for text lemmatisation with special character support.

    Features:
    - Preserves emojis and text-based smiles
    - Handles CAPS words with lemmatisation
    - Preserves punctuation context
    """

    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()

        self.morph = pymorphy3.MorphAnalyzer()

        self.stop_words = frozenset(
            set(stopwords.words(self.config.language)) | self.config.extra_stopwords
        )

        self.particles = frozenset({'не', 'ни'})

        # Initialise caches
        self._morph_cache = lru_cache(maxsize=self.config.cache_size)(self._parse)

        from collections import OrderedDict
        self._result_cache: OrderedDict = OrderedDict()

        self._caps_cache: Dict[str, str] = {}

    def _parse(self, token: str):
        """
        Parse a token using the morphological analyser.
        """
        try:
            return self.morph.parse(token)[0]
        except (ValueError, AttributeError, IndexError):
            return None

    def _get_caps_lemma(self, word: str) -> str:
        """
        Get lemma for a CAPS word with caching.
        Converts to lowercase, analyses, and returns normalised form.
        """
        if word in self._caps_cache:
            return self._caps_cache[word]

        parsed = self._morph_cache(word.lower())
        if parsed:
            lemma = parsed.normal_form
        else:
            lemma = word.lower()

        self._caps_cache[word] = lemma
        return lemma

    def process(self, text: str, keep_caps: Optional[bool] = None,
                remove_stopwords: Optional[bool] = None) -> str:
        """
        Process text through the complete lemmatisation pipeline.
        """
        text = re.sub(r'\\n|\\r|\n|\r', ' ', text)
        # Input validation
        if not text or not isinstance(text, str):
            return ""

        text = text.strip()
        if not text:
            return ""

        # Use passed parameters or fallback to config defaults
        keep_caps = keep_caps if keep_caps is not None else self.config.keep_caps
        remove_stopwords = remove_stopwords if remove_stopwords is not None else self.config.remove_stopwords

        # Check cache for this exact combination
        key = (text, keep_caps, remove_stopwords)
        if key in self._result_cache:
            # Update LRU order
            self._result_cache.move_to_end(key)
            return self._result_cache[key]

        # Process the text
        result = self._process(text, keep_caps, remove_stopwords)

        # Update LRU cache
        if len(self._result_cache) >= self.config.cache_size:
            self._result_cache.popitem(last=False)
        self._result_cache[key] = result

        return result

    def _process(self, text: str, keep_caps: bool, remove_stopwords: bool) -> str:
        """
        Internal processing pipeline for text lemmatisation.

        Steps:
        1. Emoji replacement with semantic markers
        2. Text-based smile replacement
        3. CAPS word handling with lemmatisation
        4. Punctuation processing with context preservation
        5. Number detection and marking
        6. Tokenisation
        7. Lemmatisation with POS filtering
        8. Final cleanup
        """

        text = text.replace('\n', ' ').replace('\r', ' ')

        # 1. Emoji processing
        def replace_emoji(match):
            emoji = match.group(0)
            if emoji in EMOJI_MAP:
                return f' {EMOJI_MAP[emoji]} '
            elif self.config.log_unknown_emoji:
                logger.debug(f"Unknown emoji: {emoji}")
            return f' {emoji} '

        text = RE_EMOJI.sub(replace_emoji, text)

        # 2. Smile processing
        def replace_smile(match):
            smile = match.group(0).lower()
            # Normalise: remove dash and spaces
            normalised = smile.replace('-', '').replace(' ', '')
            return f' {SMILE_MAP.get(normalised, f"SMILE_{normalised}")} '

        text = RE_SMILE.sub(replace_smile, text)

        # Handle repeated brackets as smiles
        text = re.sub(r'\){2,}', ' SMILE_BRACKET_HAPPY ', text)
        text = re.sub(r'\({2,}', ' SMILE_BRACKET_SAD ', text)

        # 3. CAPS processing with lemmatisation
        if keep_caps:
            def replace_caps(match):
                word = match.group(1)
                lemma = self._get_caps_lemma(word)
                return f' CAPS_{lemma} '
            text = RE_CAPS.sub(replace_caps, text)
        else:
            text = text.lower()

        # 4. Punctuation processing with context preservation
        # Process repeated punctuation marks
        def replace_punct(match):
            punct = match.group(0)
            if '!' in punct and '?' in punct:
                return ' PUNC_EXCLAM_QUEST '
            elif punct.count('!') >= 3:
                return ' PUNC_MANY_EXCLAM '
            elif punct.count('?') >= 3:
                return ' PUNC_MANY_QUEST '
            elif punct == '!!':
                return ' PUNC_DOUBLE_EXCLAM '
            elif punct == '??':
                return ' PUNC_DOUBLE_QUEST '
            elif '!' in punct:
                return ' PUNC_EXCLAM '
            else:
                return ' PUNC_QUEST '

        text = RE_EXCLAM_QUEST.sub(replace_punct, text)

        # Process single punctuation marks
        text = RE_SINGLE_EXCLAM.sub(' PUNC_EXCLAM ', text)
        text = RE_SINGLE_QUEST.sub(' PUNC_QUEST ', text)

        # Process ellipsis
        text = RE_ELLIPSIS.sub(' PUNC_ELLIPSIS ', text)

        # Remove other punctuation
        text = RE_OTHER_PUNCT.sub(' ', text)

        # 5. Number processing
        text = RE_NUMBER.sub(lambda m: f' {m.group(0)}_NUM ', text)

        # 6. Clean up whitespace
        text = RE_SPACE.sub(' ', text).strip()
        if not text:
            return ""

        # 7. Tokenisation
        try:
            tokens = word_tokenize(text, language=self.config.language)
        except Exception as e:
            logger.warning(f"Tokenization failed, using split: {e}")
            tokens = text.split()

        # 8. Lemmatisation and filtering
        result = []
        for token in tokens:
            if not token:
                continue

            # Preserve special tokens
            if any(token.startswith(m) for m in SPECIAL_MARKERS) or token.endswith('_NUM'):
                result.append(token)
                continue

            token_lower = token.lower()

            # Preserve particles
            if token_lower in self.particles:
                result.append(token_lower)
                continue

            # Perform morphological analysis and lemmatisation
            parsed = self._morph_cache(token_lower)
            if parsed:
                lemma = parsed.normal_form
                pos = getattr(parsed.tag, 'POS', None)

                # Filter stopwords
                if remove_stopwords and lemma in self.stop_words:
                    continue

                # Filter by POS and minimum length
                if pos in self.config.allowed_pos or len(lemma) > self.config.min_token_len:
                    result.append(lemma)
            elif len(token_lower) > self.config.min_token_len:
                # Fallback for unknown tokens
                result.append(token_lower)

        # 9. Final cleanup and filtering
        filtered = []
        for t in result:
            if len(t) > self.config.min_token_len:
                filtered.append(t)
            elif any(t.startswith(m) for m in SPECIAL_MARKERS) or t.endswith('_NUM'):
                filtered.append(t)
            elif len(t) == 1 and t.isalpha():
                filtered.append(t)

        # Join and cleanup final result
        return RE_SPACE.sub(' ', ' '.join(filtered)).strip()

    def clear_cache(self):
        """Clear all caches to free memory."""
        self._morph_cache.cache_clear()
        self._result_cache.clear()
        self._caps_cache.clear()


In [ ]:
# QUICK ACCESS FUNCTION

_lemmatiser: Optional[TextLemmatiser] = None


def lemmatise(text: str, **kwargs) -> str:
    """
    Quick convenience function for text lemmatization.
    """
    global _lemmatiser
    if _lemmatiser is None:
        _lemmatiser = TextLemmatiser()
    return _lemmatiser.process(text, **kwargs)


In [ ]:
# MAIN EXECUTION
tqdm.pandas()

df = pd.read_csv('geo-reviews-dataset-2023_11-30.csv')

# Lemmatisation with tokens
lemmatised = df['text'].progress_apply(
    lambda x: lemmatise(str(x)) if pd.notna(x) and str(x).strip() else ''
)

# Add file name for saving
df['lemmatisation_with_tokens'] = lemmatised
df.to_csv('geo-reviews-dataset-2023_11-30_lemmatised.csv', index=False)
